In [1]:
%pip install pandas numpy scipy scikit-learn


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import logging
import re


In [3]:
class MovieRecommender:
    def __init__(self, movies_path: str, ratings_path: str):
        """Initialize the recommender system with data paths."""
        self.movies_df, self.ratings_df = self._load_data(movies_path, ratings_path)
        self.genre_matrix = None
        self.user_item_matrix = None
        self.feature_matrix = None
        self.knn_model = None
        self._prepare_data()

    def _load_data(self, movies_path: str, ratings_path: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """Load and validate input datasets."""
        movies = pd.read_csv(movies_path)
        ratings = pd.read_csv(ratings_path)
        
        # Basic data validation
        assert not movies.empty, "Movies dataset is empty"
        assert not ratings.empty, "Ratings dataset is empty"
        
        return movies, ratings

    def _prepare_data(self) -> None:
        """Prepare and transform data for modeling."""
        # Extract year from movie title
        self.movies_df['year'] = self.movies_df['title'].str.extract(r'\((\d{4})\)$')[0]
        self.movies_df['year'] = pd.to_numeric(self.movies_df['year'], errors='coerce')
        self.movies_df['year'].fillna(self.movies_df['year'].median(), inplace=True)

        # Create genre matrix with year
        self._create_genre_matrix()
        
        # Create sparse user-item matrix
        self._create_user_item_matrix()
        
        # Create hybrid feature matrix
        self._create_feature_matrix()
        
        # Train KNN model
        self._train_knn_model()

    def _create_genre_matrix(self) -> None:
        """Create genre matrix with year and genre features."""
        genres = self.movies_df['genres'].str.get_dummies(sep='|')
        year_scaler = MinMaxScaler()
        scaled_year = year_scaler.fit_transform(self.movies_df[['year']])
        self.genre_matrix = pd.concat([genres, pd.DataFrame(scaled_year, columns=['scaled_year'])], axis=1)

    def _create_user_item_matrix(self) -> None:
        """Create sparse user-item matrix with implicit feedback."""
        dense_matrix = self.ratings_df.pivot_table(index='userId', columns='movieId', values='rating', fill_value=0)
        self.user_item_matrix = csr_matrix(dense_matrix.values)
        self.movie_id_to_column = {mid: idx for idx, mid in enumerate(dense_matrix.columns)}
        self.global_avg_rating = self.ratings_df['rating'].mean()

    def _create_feature_matrix(self) -> None:
        """Create hybrid feature matrix combining various signals."""
        movie_stats = self.ratings_df.groupby('movieId').agg(
            avg_rating=('rating', 'mean'),
            rating_count=('rating', 'count')
        ).reset_index()
        features = self.movies_df.merge(movie_stats, on='movieId', how='left')
        features['avg_rating'].fillna(self.global_avg_rating, inplace=True)
        features['rating_count'].fillna(0, inplace=True)
        scaler = StandardScaler()
        numeric_features = ['year', 'avg_rating', 'rating_count']
        features[numeric_features] = scaler.fit_transform(features[numeric_features])
        self.feature_matrix = pd.concat([features[numeric_features], self.genre_matrix], axis=1)

    def _train_knn_model(self, n_neighbors: int = 20) -> None:
        """Train hybrid recommendation model."""
        self.knn_model = NearestNeighbors(n_neighbors=n_neighbors, metric='cosine', algorithm='auto')
        self.knn_model.fit(self.feature_matrix)

    def find_movie(self, movie_query: str) -> Optional[pd.DataFrame]:
        """Fuzzy search for movies with interactive selection."""
        pattern = re.compile(re.escape(movie_query.lower()), re.I)
        matches = self.movies_df[self.movies_df['title'].str.contains(pattern)]
        if not matches.empty:
            return matches[['movieId', 'title', 'genres', 'year']]
        return None

    def get_recommendations(self, movie_query: str, n_recommendations: int = 10, genre_weight: float = 0.4, rating_weight: float = 0.6) -> List[Tuple[str, float]]:
        """Get hybrid recommendations for a movie."""
        matches = self.find_movie(movie_query)
        if matches is None or matches.empty:
            print("No matching movies found")
            return []

        # Automatically select the first match
        movie_id = matches.iloc[0]['movieId']

        # Find the index of the movie in the movies DataFrame
        movie_idx = self.movies_df.index[self.movies_df['movieId'] == movie_id].tolist()
        if not movie_idx:
            print("Movie ID not found in dataset")
            return []
        movie_idx = movie_idx[0]

        # Check if ratings are available for the movie
        has_ratings = movie_id in self.movie_id_to_column
        if not has_ratings:
            print("No ratings available, using content-based recommendations")
            genre_weight = 1.0
            rating_weight = 0.0

        # Generate recommendations using the KNN model
        distances, indices = self.knn_model.kneighbors(self.feature_matrix.iloc[movie_idx:movie_idx+1], n_neighbors=n_recommendations + 1)

        # Prepare recommendations
        recommendations = []
        for i, (idx, dist) in enumerate(zip(indices[0], distances[0])):
            if i == 0:
                continue  # Skip the query movie itself
            movie = self.movies_df.iloc[idx]
            score = 1 - dist  # Convert distance to similarity score
            recommendations.append((movie['title'], movie['genres'], movie['year'], score))

        # Display recommendations
        print(f"\nTop {n_recommendations} recommendations for '{matches.iloc[0]['title']}':")
        for idx, (title, genres, year, score) in enumerate(recommendations, 1):
            print(f"{idx}. {title}")
            print(f"   Genres: {genres}")
            print(f"   Similarity: {score:.3f}\n")

        return recommendations


NameError: name 'Tuple' is not defined

In [ ]:
recommender = MovieRecommender('movies.csv', 'ratings.csv')
recommender.get_recommendations("Toy Story", n_recommendations=5);
